# Lesson 10 — Spark and Distributed DataFrames

**The bridge from SQL joins.** Lessons 8–9 proved that pandas and SQL can express the same analysis. Spark adds a third spelling: a lazy DataFrame plan that can run across partitions when one process or one database file is no longer enough.

```
retail CSVs → typed Spark DataFrames → lazy transformations → action → result
                         │                         │
                         └── Parquet partitions ───┘
```

This notebook runs in local mode. It teaches the execution model without requiring a cluster or a network.


## Prerequisites and setup

- Lesson 2: DataFrames and Parquet preserve useful structure.
- Lessons 8–9: filters, joins, groups, and windows already have familiar SQL and pandas spellings.
- Install `pyspark` and a supported Java runtime (Java 17+). Opening the notebook in Jupyter runs it from this `notebooks/` folder, which the paths below assume.


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Spark workers must run the same Python as this kernel; without this pin
# they spawn whatever python3 is on PATH and fail on version skew.
os.environ.setdefault('PYSPARK_PYTHON', sys.executable)
os.environ.setdefault('PYSPARK_DRIVER_PYTHON', sys.executable)

from pyspark.sql import SparkSession, Window, functions as F

Path('../data').mkdir(exist_ok=True)
spark = (SparkSession.builder.master('local[2]').appName('intro-cs-spark')
         .config('spark.ui.enabled', 'false')
         .config('spark.sql.shuffle.partitions', '2').getOrCreate())
print('Spark:', spark.version)


## Unit 1 — A Spark DataFrame is a plan · ~30 min

A pandas operation usually runs immediately. A Spark transformation such as `select`, `filter`, or `groupBy` records a plan. An action such as `show`, `count`, `write`, or `toPandas` finally executes it. This separation lets Spark coordinate work across partitions.


In [ ]:
customers_pd = pd.read_csv('../course_data/lesson2_customers_base.csv')
transactions_pd = pd.read_csv('../course_data/lesson2_transactions_base.csv', parse_dates=['transaction_date'])
transactions_pd['transaction_year'] = transactions_pd['transaction_date'].dt.year

customers = spark.createDataFrame(customers_pd)
transactions = spark.createDataFrame(transactions_pd)
planned = transactions.filter(F.col('quantity') > 0).select('transaction_id', 'customer_id')
print('No result has been displayed yet; this is a transformation plan.')
planned.explain()
planned.show(3)  # show is an action


> **Unit 1 takeaway:** transformations describe *what* should happen; actions trigger *when* Spark carries out the plan. Avoid `collect()` unless the result is truly small enough for the driver.


## Unit 2 — The same retail answer, Spark spelling · ~45 min

The join and aggregation below answer the familiar monthly-revenue question. Compare it with the SQL and pandas versions from Lessons 8–9.


In [ ]:
monthly_revenue = (transactions.join(customers, 'customer_id')
    .withColumn('month', F.date_format('transaction_date', 'yyyy-MM'))
    .withColumn('revenue', F.col('quantity') * F.col('unit_price'))
    .groupBy('country', 'month')
    .agg(F.sum('revenue').alias('revenue'), F.count('*').alias('line_items'))
    .orderBy('country', 'month'))
monthly_revenue.show()

monthly_revenue.createOrReplaceTempView('monthly_revenue')
spark.sql('SELECT country, month, revenue FROM monthly_revenue ORDER BY country, month').show()


## Unit 3 — Windows still describe groups without collapsing them · ~30 min


In [ ]:
customer_revenue = (transactions.join(customers, 'customer_id')
    .withColumn('revenue', F.col('quantity') * F.col('unit_price'))
    .groupBy('country', 'customer_id').agg(F.sum('revenue').alias('revenue')))
rank_window = Window.partitionBy('country').orderBy(F.desc('revenue'))
(customer_revenue.withColumn('revenue_rank', F.dense_rank().over(rank_window))
 .orderBy('country', 'revenue_rank', 'customer_id').show())


> **Unit 3 takeaway:** `groupBy` collapses rows; a window calculates over a group while retaining its rows. The idea is identical to Lesson 9 SQL windows.


## Unit 4 — Parquet, partitions, and shuffles · ~35 min

Write a runtime-only Parquet dataset partitioned by year. Reading it back with a year filter gives Spark a chance to avoid unrelated partitions. This is a physical-layout choice, not a promise that this tiny local dataset will be faster.


In [ ]:
parquet_path = Path('../data/lesson10_transactions.parquet')
(transactions.write.mode('overwrite').partitionBy('transaction_year').parquet(str(parquet_path)))
partitioned = spark.read.parquet(str(parquet_path))
partitioned.filter(F.col('transaction_year') == 2010).explain()
partitioned.filter(F.col('transaction_year') == 2010).groupBy('source_country').count().show()


A `groupBy`, join, or window can require a **shuffle**: Spark redistributes rows so related keys meet in the same partition. Shuffles are often necessary, but they are costly at scale. `repartition` is a deliberate layout change; it is not a magic speed button.


## Unit 5 — Choose the smallest engine that fits · ~20 min

| Need | Best first choice | Why |
| --- | --- | --- |
| Explore a small in-memory table | pandas | Immediate, flexible analysis |
| Enforce relationships or query a local durable file | SQLite | Schema, constraints, and selective SQL reads |
| Process a dataset that needs partitioned/distributed execution | Spark | Lazy plans and distributed DataFrames |

Spark is not automatically better. Its setup, serialization, and shuffle costs are real; use it when the data or workload justifies them.


## Practice and wrap-up

Complete `lesson10_exercise/`: implement one Spark join/aggregation and one window query. The offline checker compares your answers with pinned retail results.

> **The lesson, one sentence:** Spark is the same declarative data work you know from SQL and pandas, but represented as a lazy plan that can be executed across partitions.

When you are done, stop the local session: `spark.stop()`.
